# Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn3

import io

from itertools import combinations
from scipy import stats
from scipy.stats import pearsonr

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
%matplotlib inline

In [ ]:
sns.set_theme(context='notebook', style='white', 
              font = 'arial', font_scale = 1)

## Discovery cohort data import

In [ ]:
maf = pd.read_csv("input/discovery/Discovery_rare_final_maf_maftools.maf", sep = "\t")
maf['vaf'] = maf['t_alt_count']/(maf['t_alt_count']+maf['t_ref_count'])
maf

In [ ]:
print(len(maf["Tumor_Sample_Barcode"].value_counts()))
maf['vaf'].describe()

In [ ]:
vaf_df = pd.DataFrame(maf['Tumor_Sample_Barcode'].value_counts())
vaf_df.rename(columns={'count': 'vaf_filtered_mut_count'}, inplace=True)

print(vaf_df.shape)
vaf_df['vaf_filtered_mut_count'].describe()

In [ ]:
df = pd.read_csv("./input/meta_obreg.csv")
print(df.shape)
df.head()

In [ ]:
df['Path_MVM_binary'] = (df['asw_subclass_pure_mvm'] + df['asw_subclass_mvm_fvm'] + 
                         df['asw_subclass_mvm_vue'])
df['SGA_binary'] = df['calculated_iugr.x'].map({0.0: 0, 1.0: 1})

In [ ]:
decon = pd.read_csv("input/discovery/BulkRNAseq_meta_cell_type_deconvolution_20260115_GSE182381.csv")

In [ ]:
decon['maternal_cell_type_contribution'] = (decon['Maternal.B.C'] + decon['Maternal.CD1'] + 
                                              decon['Maternal.CD8'] + decon['Maternal.FCG'] + 
                                              decon['Maternal.Nai'] + decon['Maternal.Nat'] + decon['Maternal.Pla'])

In [ ]:
df = pd.merge(df, decon[['SampleID', 'maternal_cell_type_contribution']])

In [ ]:
batch_df = pd.read_csv("input/discovery/Clinical_data_binary_genes_fullpathways_hypoxia_tmb_20240311.csv")
batch_df = batch_df[['SampleID', 'batch.x']]
batch_df.head()

In [ ]:
df = pd.merge(df, batch_df, on = 'SampleID')
df

In [ ]:
disc_merged_df = pd.merge(df, vaf_df, left_on = 'SampleID', right_on = 'Tumor_Sample_Barcode')
disc_merged_df['eTMB_VAF_filtered'] = disc_merged_df['vaf_filtered_mut_count']*100/disc_merged_df['Number_of_bases_per106.megabase.']
disc_merged_df['eTMB'] = disc_merged_df['eTMB_VAF_filtered']
disc_merged_df.shape

In [ ]:
disc_merged_df.head()

In [ ]:
hyp = pd.read_csv("input/discovery/Disc_hypoxia_gene_scoring_20260914.csv")
hyp.head()

In [ ]:
hyp = hyp.iloc[:, 0:2]
hyp.columns = ['SampleID', 'HypoxiaScoreBuffa_new']


In [ ]:
disc_merged_df2 = pd.merge(disc_merged_df, hyp, left_on = 'SampleID', right_on = 'SampleID')
disc_merged_df2

In [ ]:
disc_merged_df2['Path_MVM_binary'].value_counts()

In [ ]:
disc_merged_df2.to_csv("output/discovery/Disc_cohort_metadata_20260913.csv")

In [ ]:
df = disc_merged_df2
df['variant_cohort'].value_counts()

## Validation cohort data import

In [ ]:
maf = pd.read_csv("input/validation/Validation_rare_final_maf_maftools.maf", sep = "\t")
maf['vaf'] = maf['t_alt_count']/(maf['t_alt_count']+maf['t_ref_count'])
maf

In [ ]:
print(len(maf["Tumor_Sample_Barcode"].value_counts()))
maf['vaf'].describe()

In [ ]:
vaf_df = pd.DataFrame(maf['Tumor_Sample_Barcode'].value_counts())
vaf_df.rename(columns={'count': 'vaf_filtered_mut_count'}, inplace=True)

print(vaf_df.shape)
vaf_df['vaf_filtered_mut_count'].describe()

In [ ]:
df = pd.read_csv("./input/meta_obreg.csv",index_col=0)
print(df.shape)
df.head()

In [ ]:
df['Path_MVM_binary'] = (df['asw_subclass_pure_mvm'] + df['asw_subclass_mvm_fvm'] + 
                         df['asw_subclass_mvm_vue'])
df['SampleID'] = 'Sample' + df.index.astype(str)

In [ ]:
val_merged_df = pd.merge(df, vaf_df, left_on = 'SampleID', right_on = 'Tumor_Sample_Barcode')
val_merged_df['eTMB_VAF_filtered'] = val_merged_df['vaf_filtered_mut_count']*100/val_merged_df['Number_of_bases_per106.megabase.']
val_merged_df['eTMB'] = val_merged_df['eTMB_VAF_filtered']
val_merged_df.shape

In [ ]:
val_merged_df['Path_MVM_binary'].value_counts()

In [ ]:
hyp = pd.read_csv("input/validation/Val_hypoxia_gene_scoring_20260914.csv")
hyp = hyp.iloc[:, 0:2]
hyp.columns = ['SampleID', 'HypoxiaScoreBuffa_new']


In [ ]:
val_merged_df2 = pd.merge(val_merged_df, hyp, left_on = 'SampleID', right_on = 'SampleID')

In [ ]:
val_merged_df2.to_csv("output/validation/Val_cohort_metadata_20260913.csv")

# Figure and Extended Data Figure 1

In [ ]:
disc_merged_df = pd.read_csv("output/discovery/Disc_cohort_metadata_20260913.csv")

## Distributions

In [ ]:
df = disc_merged_df
df['variant_cohort'].value_counts()

In [ ]:
print(df[df['Path_MVM_binary'] == 1]['PE_binary'].value_counts())
print(df[df['asw_subclass_nonlesion'] == 1]['PE_binary'].value_counts())
print(df[df['asw_subclass_pure_fvm'] == 1]['PE_binary'].value_counts())
print(df[df['asw_subclass_pure_vue'] == 1]['PE_binary'].value_counts())

In [ ]:
print(df[df['calculated_iugr.x'] == 0]['PE_binary'].value_counts())
print(df[df['calculated_iugr.x'] == 1]['PE_binary'].value_counts())

In [ ]:
maf = pd.read_csv("input/discovery/discovery_rare_final_maf_maftools.maf", sep = "\t")
maf

In [ ]:
len(maf['Hugo_Symbol'].unique())

In [ ]:
maf = pd.read_csv("input/discovery/Discovery_rare_damaging_only_final_maf_maftools.maf", sep = "\t")
maf

In [ ]:
len(maf['Hugo_Symbol'].unique())

## Figure 1A, ED 1B (Demographic plots):

In [ ]:
test_df = df

In [ ]:
print("No PE Mean:", test_df[test_df['PE_binary'] == 0]['age_delivery.x'].mean())
print("PE Mean:", test_df[test_df['PE_binary'] == 1]['age_delivery.x'].mean())

In [ ]:
test_df['MatAgeBin'] = np.where((test_df['age_delivery.x']>=35) , 1, 0)
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}

sns.set_style("ticks")

plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="MatAgeBin", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors, hue="PE_onset_uniform")

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Maternal age at delivery')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['MatAgeBin'] = np.where((test_df['age_delivery.x']>=35) , 1, 0)
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}

sns.set_style("ticks")

plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="MatAgeBin", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors, hue="PE_onset_uniform")

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Maternal age at delivery')
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_Placental_Maternal_Age_20250707.pdf", format="pdf", bbox_inches="tight")

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="Path_MVM_binary", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Maternal vascular malperfusion')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="Path_MVM_binary", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Maternal vascular malperfusion')
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_MVM_20250707.pdf", format="pdf", bbox_inches="tight")

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")

plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="calculated_iugr.x", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Small for gestational age')
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_SGA_20250707.pdf", format="pdf", bbox_inches="tight")

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")

plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="calculated_iugr.x", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

ax.set_ylabel('')
ax.set_yticks([]) 
plt.title('Small for gestational age')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="Fetal_sex_binary", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)


plt.title('Fetal sex')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="Fetal_sex_binary", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)


plt.title('Fetal sex')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_Fetal_sex_binary_20250707.pdf", format="pdf", bbox_inches="tight")

In [ ]:
test_df['GesAgeBin'] = np.where((test_df['GA']>=34) , 0, 1)
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = {
    0: "white",
    1: "#ECECEC"
}
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="GesAgeBin", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors, hue="PE_onset_uniform") # color=".2", size = 3.5)

plt.title('Gestational age')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['GesAgeBin'] = np.where((test_df['GA']>=34) , 0, 1)
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = {
    0: "white",
    1: "#ECECEC"
}
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}
sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="GesAgeBin", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors, hue="PE_onset_uniform") # color=".2", size = 3.5)

plt.title('Gestational age')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_GA_20250707.pdf", format="pdf", bbox_inches="tight")

In [ ]:
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}

sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="itpt_delivery_mode", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

plt.title('Delivery mode')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

In [ ]:
test_df['GesAgeBin'] = np.where((test_df['GA']>=34) , 1, 0)
test_df['PE_names'] = np.where((test_df['PE_binary']==0) , "Normotensive", "Preeclampsia")
test_df.sort_values(by='PE_names', ascending=True, inplace=True)

colors1 = ["white", "#ECECEC"]
colors = {
    "Normal_EO": "#508A91",
    "Normal_LO": "#B1CCC2",
    "PE_LO": "#B4A8C7",
    "PE_EO": "#6666A3"
}

sns.set_style("ticks")
plt.figure()
ax = sns.violinplot(data=test_df, x="GA", y="PE_names", hue="itpt_delivery_mode", fill=True,
                       split=True, scale="count", scale_hue=False, palette=colors1, inner="quartile", 
                      orient="h")
sns.swarmplot(x="GA", y="PE_names", data=test_df, palette=colors,hue="PE_onset_uniform") # color=".2", size = 3.5)

plt.title('Delivery mode')
ax.set_ylabel('')
ax.set_yticks([]) 
plt.legend().remove()
plt.xlim(17, 48)
plt.xlabel('')
ax.tick_params(axis='x', left=False, length=2, width=1)
ax.xaxis.label.set_color('black')

plt.savefig("output/figure1/Demographics_Delivery_mode_20250707.pdf", format="pdf", bbox_inches="tight")

## Figure 1A, ED 1B (Chi-squared):

In [ ]:
from tableone import TableOne

In [ ]:
test_df = disc_merged_df

In [ ]:
# test for normality 
stat, p_value = stats.shapiro(test_df['GA'])
p_value

In [ ]:
stat, p_value = stats.shapiro(test_df['age_delivery.x'])
p_value

In [ ]:
test_df['SGA_text'] = test_df['calculated_iugr.x'].map({0: 'Normal', 1: 'SGA'}).astype(str)
test_df['Fetal_sex_text'] = test_df['Fetal_sex_binary'].map({0: 'Male', 1: 'Female'}).astype(str)
test_df['Delivery_text'] = test_df['itpt_delivery_mode'].map({1.0: 'Caesarean', 2: 'Vaginal'}).astype(str)
test_df['MVM_text'] = test_df['Path_MVM_binary'].map({0: 'Non-MVM', 1: 'MVM'}).astype(str)
test_df['PE_text'] = test_df['PE_binary'].map({0: 'Normotensive', 1: 'Preeclampsia'}).astype(str)

# columns to summarize
columns = ["GA", 'age_delivery.x', "Fetal_sex_text", 
           "SGA_text", 'Delivery_text', "MVM_text"]

# columns containing categorical variables
categorical = ["Fetal_sex_text", "SGA_text", 'Delivery_text', "MVM_text"]

# non-normal variables
nonnormal = ["GA"]

# limit the binary variable "death" to a single row
limit = {"Fetal_sex_text": 1,
         'SGA_text': 1,
         'Delivery_text': 1,
         'MVM_text' : 1}

# alternative labels
labels={'GA': 'Gestational age at delivery',
       'Fetal_sex_text': 'Fetal sex',
       'SGA_text': "Small for gestational age",
       'Delivery_text': "Mode of delivery",
        "MVM_text": "Presence of maternal vascular malperfusion"
       }

# optionally, a categorical variable for stratification
groupby = "PE_text"

# display minimum and maximum for listed variables
min_max = ['GA', 'age_delivery.x']

table2 = TableOne(test_df, columns=columns, categorical=categorical, groupby=groupby,
                  nonnormal=nonnormal,label_suffix=True,pval = True, smd=False,
                  htest_name=True, rename=labels, limit=limit,
                   min_max=min_max, pval_adjust="bonferroni")

# # Save to Excel
# fn1 = 'output/figure1/Demographics_Table_stats_PE_20261007.xlsx'
# table2.to_excel(fn1)

# # Save table to HTML
# fn3 = 'output/figure1/Demographics_Table_stats_PE_20261007.html'
# table2.to_html(fn3)

table2

In [ ]:
test_df['SGA_text'] = test_df['calculated_iugr.x'].map({0: 'Normal', 1: 'SGA'}).astype(str)
test_df['Fetal_sex_text'] = test_df['Fetal_sex_binary'].map({0: 'Male', 1: 'Female'}).astype(str)
test_df['Delivery_text'] = test_df['itpt_delivery_mode'].map({1.0: 'Caesarean', 2: 'Vaginal'}).astype(str)
test_df['MVM_text'] = test_df['Path_MVM_binary'].map({0: 'Non-MVM', 1: 'MVM'}).astype(str)
test_df['PE_text'] = test_df['PE_binary'].map({0: 'Normotensive', 1: 'Preeclampsia'}).astype(str)

# columns to summarize
columns = ["GA", 'age_delivery.x', "Fetal_sex_text", 
           "SGA_text", 'Delivery_text', 
           "PE_text"]

# columns containing categorical variables
categorical = ["Fetal_sex_text", "SGA_text", 'Delivery_text', "PE_text"]

# non-normal variables
nonnormal = ["GA"]

# limit the binary variable "death" to a single row
limit = {"Fetal_sex_text": 1,
         'SGA_text': 1,
         'Delivery_text': 1,
         'PE_text' : 1}

# alternative labels
labels={'GA': 'Gestational age at delivery',
       'Fetal_sex_text': 'Fetal sex',
       'SGA_text': "Small for gestational age",
       'Delivery_text': "Mode of delivery",
        "PE_text": "Preeclampsia"
       }

# for stratification
groupby = "MVM_text"

# display minimum and maximum for listed variables
min_max = ['GA']

table2 = TableOne(test_df, columns=columns, categorical=categorical, groupby=groupby,
                  nonnormal=nonnormal,label_suffix=True,pval = True, smd=False, 
                  htest_name=True, rename=labels, limit=limit, min_max=min_max, pval_adjust="bonferroni")

# # Save to Excel
# fn1 = 'output/figure1/Demographics_Table_stats_MVM_20261007.xlsx'
# table2.to_excel(fn1)

# # Save table to HTML
# fn3 = 'output/figure1/Demographics_Table_stats_MVM_20261007.html'
# table2.to_html(fn3)

table2

### Validation

In [ ]:
test_df = pd.read_csv("output/validation/Val_cohort_metadata_20260913.csv")

In [ ]:
# test for normality 
stat, p_value = stats.shapiro(test_df['GA'])
p_value

In [ ]:
stat, p_value = stats.shapiro(test_df['age_delivery.x'])
p_value

In [ ]:
test_df['SGA_text'] = test_df['calculated_iugr.x'].map({0: 'Normal', 1: 'SGA'}).astype(str)
test_df['Fetal_sex_text'] = test_df['Fetal_sex_binary'].map({0: 'Male', 1: 'Female'}).astype(str)
test_df['Delivery_text'] = test_df['itpt_delivery_mode'].map({1.0: 'Caesarean', 2: 'Vaginal'}).astype(str)
test_df['MVM_text'] = test_df['Path_MVM_binary'].map({0: 'Non-MVM', 1: 'MVM'}).astype(str)
test_df['PE_text'] = test_df['PE_binary'].map({0: 'Normotensive', 1: 'Preeclampsia'}).astype(str)

# columns to summarize
columns = ["GA", 'age_delivery.x', "Fetal_sex_text", 
           "SGA_text", 'Delivery_text', "MVM_text"]

# columns containing categorical variables
categorical = ["Fetal_sex_text", "SGA_text", 'Delivery_text', "MVM_text"]

# non-normal variables
nonnormal = ["GA"]

# limit the binary variable "death" to a single row
limit = {"Fetal_sex_text": 1,
         'SGA_text': 1,
         'Delivery_text': 1,
         'MVM_text' : 1}

# alternative labels
labels={'GA': 'Gestational age at delivery',
       'Fetal_sex_text': 'Fetal sex',
       'SGA_text': "Small for gestational age",
       'Delivery_text': "Mode of delivery",
        "MVM_text": "Presence of maternal vascular malperfusion"
       }

# optionally, a categorical variable for stratification
groupby = "PE_text"

# display minimum and maximum for listed variables
min_max = ['GA', 'age_delivery.x']

table2 = TableOne(test_df, columns=columns, categorical=categorical, groupby=groupby,
                  nonnormal=nonnormal,label_suffix=True,pval = True, smd=False,
                  htest_name=True, rename=labels, limit=limit,
                   min_max=min_max, pval_adjust="bonferroni")

# Save to Excel
fn1 = 'output/figure1/Val_Demographics_Table_stats_PE_20261007.xlsx'
table2.to_excel(fn1)

# Save table to HTML
fn3 = 'output/figure1/Val_Demographics_Table_stats_PE_20261007.html'
table2.to_html(fn3)

table2

In [ ]:
test_df['SGA_text'] = test_df['calculated_iugr.x'].map({0: 'Normal', 1: 'SGA'}).astype(str)
test_df['Fetal_sex_text'] = test_df['Fetal_sex_binary'].map({0: 'Male', 1: 'Female'}).astype(str)
test_df['Delivery_text'] = test_df['itpt_delivery_mode'].map({1.0: 'Caesarean', 2: 'Vaginal'}).astype(str)
test_df['MVM_text'] = test_df['Path_MVM_binary'].map({0: 'Non-MVM', 1: 'MVM'}).astype(str)
test_df['PE_text'] = test_df['PE_binary'].map({0: 'Normotensive', 1: 'Preeclampsia'}).astype(str)

# columns to summarize
columns = ["GA", 'age_delivery.x', "Fetal_sex_text", 
           "SGA_text", 'Delivery_text', 
           "PE_text"]

# columns containing categorical variables
categorical = ["Fetal_sex_text", "SGA_text", 'Delivery_text', "PE_text"]

# non-normal variables
nonnormal = ["GA"]

# limit the binary variable "death" to a single row
limit = {"Fetal_sex_text": 1,
         'SGA_text': 1,
         'Delivery_text': 1,
         'PE_text' : 1}

# alternative labels
labels={'GA': 'Gestational age at delivery',
       'Fetal_sex_text': 'Fetal sex',
       'SGA_text': "Small for gestational age",
       'Delivery_text': "Mode of delivery",
        "PE_text": "Preeclampsia"
       }

# for stratification
groupby = "MVM_text"

# display minimum and maximum for listed variables
min_max = ['GA']

table2 = TableOne(test_df, columns=columns, categorical=categorical, groupby=groupby,
                  nonnormal=nonnormal,label_suffix=True,pval = True, smd=False, 
                  htest_name=True, rename=labels, limit=limit, min_max=min_max, pval_adjust="bonferroni")

# Save to Excel
fn1 = 'output/figure1/Val_Demographics_Table_stats_MVM_20261007.xlsx'
table2.to_excel(fn1)

# Save table to HTML
fn3 = 'output/figure1/Val_Demographics_Table_stats_MVM_20261007.html'
table2.to_html(fn3)

table2

## Figure 1B (Boxplot - PE, genomic instability)

In [ ]:
test_df = disc_merged_df
sns.set_style("white") 

colors = ['#81ABB0', '#8D87B5']

ax = sns.violinplot(x = 'PE_binary', y = "eTMB", data = test_df, 
                order=[0, 1],
                boxprops={'alpha': 1}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="PE_binary", y="eTMB",
              hue="PE_binary", hue_order=[0, 1],
              order=[0, 1], 
              dodge=False, ax=ax,palette='dark:black',alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title='PE_binary', bbox_to_anchor=(1, 1.02), loc='upper left')
ax.set_xticklabels(["Normotensive", "Preeclamptic"])

plt.legend().remove()
plt.xlabel('')
plt.ylabel('Expressed rare variant burden')
plt.ylim(-0, 5)

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.savefig("output/figure1/Discovery_PE_eMB.pdf", format="pdf", bbox_inches="tight")

## Figure 1C (Boxplot - Maternal outcomes, genomic instability):

In [ ]:
test_df = disc_merged_df
sns.set_style("white") 

colors = ['#508A91', '#B1CCC2', '#6666A3','#B4A8C7']

ax = sns.violinplot(x = 'PE_onset_uniform', y = "eTMB", data = test_df, 
                order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
                boxprops={'alpha': 1}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="PE_onset_uniform", y="eTMB",
              hue="PE_onset_uniform", hue_order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
              order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"], 
              dodge=False, ax=ax, palette='dark:black', alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title='PE_onset_uniform', bbox_to_anchor=(1, 1.02), loc='upper left')
ax.set_xticklabels(["EP-N", "NTT-N", "EP-PE", "NTT-PE"])

plt.legend().remove()
plt.xlabel('Maternal outcome')
plt.ylabel('Expressed rare variant burden')
plt.ylim(-0, 5)

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.savefig("output/figure1/Discovery_PE-GA_eMB.pdf", format="pdf", bbox_inches="tight")

# Extended Data Figure 2B-D (Variant metrics)

## ED 2B (VAF):

In [ ]:
maf = pd.read_csv("input/discovery/discovery_rare_final_maf_maftools.maf", sep = "\t")
maf['vaf'] = maf['t_alt_count']/(maf['t_alt_count']+maf['t_ref_count'])

filtered_df = maf
filtered_df

In [ ]:
bins = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0] # Custom bin edges
labels = ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"]
binned_data = pd.cut(filtered_df['vaf'], bins=bins, labels=labels)
print(pd.cut(filtered_df['vaf'], bins=bins).value_counts())
print(filtered_df['vaf'].describe())

In [ ]:
plot_df = pd.Series(binned_data.value_counts(), name='count').sort_index().reset_index()
plot_df['vaf'] = plot_df['vaf'].cat.add_categories(['1.0'])
new_row = pd.DataFrame({'vaf': ['1.0'], 'count': [0]})
plot_df = pd.concat([plot_df, new_row], ignore_index=True)
plot_df

In [ ]:
maf2 = pd.read_csv("input/discovery/Discovery_rare_damaging_only_final_maf_maftools.maf", sep = "\t")
maf2['vaf'] = maf2['t_alt_count']/(maf['t_alt_count']+maf['t_ref_count'])

In [ ]:
bins = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0] # Custom bin edges
labels = ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"]
binned_data = pd.cut(maf2['vaf'], bins=bins, labels=labels)
print(pd.cut(maf2['vaf'], bins=bins).value_counts())
print(maf2['vaf'].describe())

In [ ]:
plot_df2 = pd.Series(binned_data.value_counts(), name='count_damaging').sort_index().reset_index()
plot_df2['vaf'] = plot_df2['vaf'].cat.add_categories(['1.0'])
new_row = pd.DataFrame({'vaf': ['1.0'], 'count_damaging': [0]})
plot_df2 = pd.concat([plot_df2, new_row], ignore_index=True)
df = pd.concat([plot_df, plot_df2['count_damaging']], ignore_index=True, axis = 1)
df.columns = ['VAF', 'rare', 'rare_damaging']
df

In [ ]:
fig, ax = plt.subplots()

x = np.arange(len(df))
width = 0.5  # each bar takes half the bin

ax.bar(x, df['rare'], width=width, align='edge',
       color="#76B7BD", label='rare')

ax.bar(x + width, df['rare_damaging'], width=width, align='edge',
       color="#447273", label='rare_damaging')

ax.set_xticks(x)  # center tick between the two bars
ax.set_xticklabels(df['VAF'])
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(0, 1300)
ax.set_xlabel('Variant allele frequency')
ax.set_ylabel('Number of rare somatic variants')

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.tight_layout()
plt.savefig("output/figure1/Discovery_VAF.pdf", format="pdf", bbox_inches="tight")

## ED 2C (% Heterozygous)

In [ ]:
maf = pd.read_csv("input/discovery/discovery_rare_final_maf_maftools.maf", sep = "\t")
maf = maf[maf['Chromosome'] != 'chrM']

In [ ]:
maf['t_GT'].value_counts()

In [ ]:
def genotype(gt):
    if gt in ['0/1', '0|1']:
        return 'Heterozygous'
    else:
        return 'Homozygous'

In [ ]:
maf.loc[:, 'zygosity'] = maf['t_GT'].apply(genotype)

In [ ]:
het_counts = maf.groupby('Tumor_Sample_Barcode')['zygosity'].value_counts().unstack(fill_value=0)
het_counts['percent_het'] = het_counts['Heterozygous']/(het_counts['Heterozygous']+ het_counts['Homozygous']) * 100
het_counts['percent_hom'] =  het_counts['Homozygous']/(het_counts['Heterozygous']+ het_counts['Homozygous']) * 100
het_counts['sample'] = het_counts.index

In [ ]:
het_counts

In [ ]:
fig, ax = plt.subplots()

plt.bar(het_counts['sample'], het_counts['percent_het'], color='black', edgecolor='white', linewidth = 1)
ax.set_xlabel('')

plt.ylabel('Heterozygous genotypes (%)')
plt.title('Percentage of heterozygous genotypes per sample')
plt.ylim(0, 100)
ax.grid(False)
ax.set_facecolor('white')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
plt.xticks([])
ax.tick_params(colors='black') 
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=False, left=True)

plt.tight_layout()
plt.savefig("output/figure1/Discovery_heterozygous.pdf", format="pdf", bbox_inches="tight")
plt.show()

## ED 2D (Ti-Tv)

In [ ]:
ti_tv = pd.read_csv("input/discovery/Discovery_titv.csv")
ti_tv.head()

In [ ]:
ti_tv = ti_tv[['TiTv.fractions.Ti', 'TiTv.fractions.Tv']]
ti_tv.columns = ['Ti', 'Tv']
test_df = ti_tv[['Ti', 'Tv']].melt(var_name='TiTv', value_name='Percent')

In [ ]:
fig, ax = plt.subplots()

sns.set_style("white") 
colors = ['#EEE5D8','#c56f87'] 

ax = sns.violinplot(x = "TiTv", y = "Percent", data = test_df, 
                boxprops={'alpha': 0.9}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="TiTv", y="Percent",
              dodge=True, ax = ax,palette='dark:black',alpha=0.5)
ax.set_xticklabels(["Transitions", "Transversions"])

plt.ylim(-5, 105)
plt.ylabel("Variants (%)")
plt.xlabel("")
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(colors='black') 
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)


plt.savefig("output/ed2/Discovery_TiTv.pdf", format="pdf", bbox_inches="tight")

In [ ]:
ti_tv['TiTv'] = ti_tv['Ti']/ti_tv['Tv']

In [ ]:
ti_tv['TiTv'].describe()

# Extended Data Figure 3A-D (WGS validation)

In [ ]:
files = ['hc-rna_mutect2-wgs.txt', 'hc-rna_hc-wgs.txt', 'placenta_quality_hc-rna_placenta_quality_hc-wgs.txt']
titles = ["HaplotypeCaller (RNA-seq) vs Mutect2 (WGS)", 
          "HaplotypeCaller (RNA-seq vs WGS)", 
          "Quality filtered: HaplotypeCaller (RNA-seq vs WGS)"]


In [ ]:
for file in files:
    df = pd.read_table("input/wgs_rna/" + file)
    
    df["File"] = df["Folder"].str[:4]
    df['Perc_RNA_Shared'] = 100-df['Perc_RNA']
    print(file)
    print(df['Perc_RNA'].describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4))
axes = axes.flatten()

for i, file in enumerate(files[:3]):
    ax = axes[i]

    df = pd.read_table("input/wgs_rna/" + file)
    df["File"] = df["Folder"].str[:4]
    df["Perc_RNA_Shared"] = 100 - df["Perc_RNA"]

    ax.bar(df["File"], df["Perc_RNA"], label="RNA and WGS", color="#c56f87")

    ax.bar(df["File"], df["Perc_RNA_Shared"], bottom=df["Perc_RNA"], label="RNA", color="#EEE5D8")
    
    ax.set_title(titles[i], color="black", fontsize=14)
    ax.set_ylabel("Overlap with RNA-seq variants (%)", fontsize=14)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Placenta ID", fontsize=14)

    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1)

    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.tick_params(axis="x", colors="black", labelrotation=90,
                   direction="out", length=2, width=1, bottom=True, labelsize=11)
    ax.tick_params(axis="y", colors="black", 
                   direction="out", length=2, width=1, left=True, labelsize=11)


plt.savefig("output/ed3/compare_variant_calls.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
files = os.listdir("input/correlation/")
files.sort()
files

In [ ]:
for file in files:  
    vaf_df = pd.read_csv("input/correlation/" + file)
    print(file[:4])
    r, p = pearsonr(vaf_df["vaf_x"], vaf_df["vaf_y"])
    print(r, p)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, file in enumerate(files[:8]):
    ax = axes[i]

    vaf_df = pd.read_csv("input/correlation/" + file)

    sns.regplot(x="vaf_x", y="vaf_y", data=vaf_df, ax=ax, 
                scatter_kws={"color": "black", "s": 20, "alpha": 0.5}, 
                line_kws={"color": "#A9A9A9"}, marker="o")

    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1)

    ax.set_xlabel("Sum of RNA-seq VAF for each gene", fontsize=17, color="black")
    ax.set_ylabel("Sum of WGS VAF for each gene", fontsize=17, color="black")

    ax.tick_params(
        direction="out",
        length=2,
        width=1,
        colors="black",
        bottom=True,
        left=True,
        labelsize=15
    )

    ax.set_title(file[:4], fontsize=17, color="black")

plt.tight_layout()

plt.savefig(
    "output/ed3/gene_vaf.tiff",
    format="tiff",
    bbox_inches="tight",
    dpi=400
)

plt.show()

# Extended Data Figure 5 A-D

In [ ]:
test_df = disc_merged_df

## ED5A

In [ ]:
sns.set_style("white") 
colors =  ['#508A91', '#B1CCC2', '#6666A3','#B4A8C7']


ax =sns.boxplot(x = 'PE_onset_uniform', y = "maternal_cell_type_contribution", data = test_df, 
                order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
                boxprops={'alpha': 1}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="PE_onset_uniform", y="maternal_cell_type_contribution",
              hue="PE_onset_uniform", hue_order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
              order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"], 
              dodge=False, ax=ax,palette='dark:black',alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title='PE_onset_uniform', bbox_to_anchor=(1, 1.02), loc='upper left')
ax.set_xticklabels(["EP-N", "NTT-N", "EP-PE", "NTT-PE"])
plt.legend().remove()
plt.xlabel('Maternal outcomes')
plt.ylabel('Maternal cell type contribution (%)')
ax.set_ylim(-1, 20)
ax.set_yticks([0, 5, 10, 15, 20])

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)

plt.savefig("output/ed4/Discovery_maternal_cell_type_contribution.pdf", format="pdf", bbox_inches="tight")

## ED5B

In [ ]:
g = sns.lmplot(x="eTMB", y="maternal_cell_type_contribution", data=test_df,
               scatter_kws={"color": "black", "s": 5},
               line_kws={"color": "gray"})
ax = g.ax 

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)


ax.set_xlabel("Expressed rare variant burden")
ax.set_ylabel("Maternal cell type contribution (%)")

plt.savefig("output/ed4/Discovery_maternal_cell_type_contribution_by_eMB.pdf", format="pdf", bbox_inches="tight")
plt.savefig("output/ed4/Discovery_maternal_cell_type_contribution_by_eMB.tiff", dpi = 300, bbox_inches="tight")

In [ ]:
r, p = pearsonr(test_df["eTMB"], test_df["maternal_cell_type_contribution"])
print(r, p)

## ED5C

In [ ]:
g = sns.lmplot(x="HypoxiaScoreBuffa_new", y="maternal_cell_type_contribution", data=test_df,
               scatter_kws={"color": "black", "s": 5},
               line_kws={"color": "gray"})
ax = g.ax 

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)


ax.set_xlabel("Hypoxia response score")
ax.set_ylabel("Maternal cell type contribution (%)")

plt.savefig("output/ed4/Discovery_maternal_cell_type_contribution_by_hypoxia.pdf", format="pdf", bbox_inches="tight")
plt.savefig("output/ed4/Discovery_maternal_cell_type_contribution_by_hypoxia.tiff", dpi = 300, bbox_inches="tight")

In [ ]:
r, p = pearsonr(test_df["HypoxiaScoreBuffa_new"], test_df["maternal_cell_type_contribution"])
print(r, p)

## ED5D

In [ ]:
g = sns.lmplot(x="HypoxiaScoreBuffa_new", y="GA", data=test_df,
               scatter_kws={"color": "black", "s": 5},
               line_kws={"color": "gray"})
ax = g.ax 

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)


ax.set_xlabel("Hypoxia response score")
ax.set_ylabel("Gestational age")

plt.savefig("output/ed4/Discovery_HypoxiaScoreBuffa_new_by_GA.pdf", format="pdf", bbox_inches="tight")
plt.savefig("output/ed4/Discovery_HypoxiaScoreBuffa_new_by_GA.tiff", dpi = 300, bbox_inches="tight")

In [ ]:
r, p = pearsonr(test_df["GA"], test_df["HypoxiaScoreBuffa_new"])
print(r, p)

# Figure 3

## Figure 3B

In [ ]:
test_df = disc_merged_df

sns.set_style("white") 
colors =  ['#508A91', '#B1CCC2', '#6666A3','#B4A8C7']


ax =sns.violinplot(x = 'PE_onset_uniform', y = "HypoxiaScoreBuffa_new", data = test_df, 
                order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
                boxprops={'alpha': 1}, palette = colors, showfliers=False)
sns.stripplot(data=test_df, x="PE_onset_uniform", y="HypoxiaScoreBuffa_new",
              hue="PE_onset_uniform", hue_order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"],
              order=["Normal_EO", "Normal_LO", "PE_EO", "PE_LO"], 
              dodge=False, ax=ax,palette='dark:black',alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title='PE_onset_uniform', bbox_to_anchor=(1, 1.02), loc='upper left')
ax.set_xticklabels(["EP-N", "NTT-N", "EP-PE", "NTT-PE"])
plt.legend().remove()
plt.xlabel('Maternal outcomes')
plt.ylabel('Hypoxia response score')
plt.ylim(-60, 75)

for spine in ax.spines.values():
    spine.set_color((0, 0, 0, 1.0))
    spine.set_linewidth(1)

ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')
ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)

plt.savefig("output/figure3//Discovery_HypoxiaScoreBuffa.pdf", format="pdf", bbox_inches="tight")

## Figure 3C

In [ ]:
disc_maf = pd.read_csv("input/discovery/Discovery_rare_damaging_only_final_maf_maftools.maf", sep = "\t")
disc_maf = disc_maf[disc_maf['Variant_Type'] == 'SNP']
disc_maf

In [ ]:
val_maf = pd.read_csv("input/validation/Validation_rare_damaging_only_final_maf_maftools.maf", sep = "\t")
val_maf = val_maf[val_maf['Variant_Type'] == 'SNP']
val_maf

In [ ]:
circ_meta = pd.read_csv("output/circulating/circulating_cohort_metadata_20260913.csv", index_col = 0)
meta = circ_meta[circ_meta['sampling_order'].isin([1])]
m1_1 = meta[meta['Batch'] == 'Discovery']
m1_1['Tumor_Sample_Barcode'] = 'Sample' + m1_1.index
m1_1.head()

In [ ]:
maf1 = pd.read_csv("input/circulating/Circulating_Discovery_rare_damaging_only_final_maf_maftools.maf", 
                  sep = "\t")
maf1[maf1['Tumor_Sample_Barcode'].isin(m1_1['Tumor_Sample_Barcode'])]
maf1 = maf1[maf1['Variant_Type'] == 'SNP']
maf1

In [ ]:
fig, ax = plt.subplots()

v = venn3([set(disc_maf['Hugo_Symbol']), 
       set(val_maf['Hugo_Symbol']), 
       set(maf1['Hugo_Symbol'])], 
      set_labels=('Discovery', 'Validation', 'Circulating'), 
      set_colors=('#9bbbbc', '#6e5053', '#EEE5D8'), alpha = 0.7)

plt.savefig("output/figure3/All_venn.pdf", format="pdf", bbox_inches="tight")
plt.show()

# Figure 4 (Single-cell RNA-seq)

## Figure 4A

In [ ]:
df = pd.read_csv("input/single_cell/meta_all_eMB.csv")
df['eMB'].describe()

In [ ]:
df["scale_eMB"] = np.log1p(df["eMB"])

In [ ]:
palette = {
    "Immune" : "#8FA8C7",
    "Endothelial" : "#5B6770",
    "Fibroblast" : "#D6B47C",
    "Cytotrophoblast" : "#B8A1C8",
    "Extravillous trophoblast" : "#C99A9A"
}

In [ ]:
ax = sns.violinplot(x="cell_type", y="scale_eMB", data=df,
                    order = ['Cytotrophoblast',  'Extravillous trophoblast', 
                             'Fibroblast', 'Immune', 'Endothelial'], inner = None, palette=palette)
sns.stripplot(x="cell_type", y="scale_eMB", data=df, 
              order = ['Cytotrophoblast',  'Extravillous trophoblast', 
                             'Fibroblast', 'Immune', 'Endothelial'],
              size= 0.5, color= "black", dodge = True, alpha = 0.25)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.xlabel('')
plt.ylabel('Expressed rare variant burden (log1p)')
plt.ylim(-1, 15.2)
# ax.set_xticklabels([])
plt.show()

In [ ]:
ax = sns.violinplot(x="cell_type", y="scale_eMB", data=df,
                    order = ['Cytotrophoblast',  'Extravillous trophoblast', 
                             'Fibroblast', 'Immune', 'Endothelial'], inner = None, palette=palette)
sns.stripplot(x="cell_type", y="scale_eMB", data=df, 
              order = ['Cytotrophoblast',  'Extravillous trophoblast', 
                             'Fibroblast', 'Immune', 'Endothelial'],
              size= 0.5, color= "black", dodge = True, alpha = 0.25)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.xlabel('')
plt.ylabel('Expressed rare variant burden (log1p)')
plt.ylim(-1, 15.2)
ax.set_xticklabels([])
plt.savefig("output/figure4/All_eMB_vstroph.tiff", format="tiff", bbox_inches="tight", dpi = 600)
plt.show()

## Figure 4B

In [ ]:
df = pd.read_csv("input/single_cell/troph_eMB_20260920.csv")
df['cell_types'] = np.where(df['cell_type'] == "CTB", 'Cytotrophoblast', 'Extravillous trophoblast')

In [ ]:
df['cell_types'].value_counts()

In [ ]:
df["scale_eMB"] = np.log1p(df["eMB"])

In [ ]:
colors = ['#B1CCC2', '#6666A3']
ax = sns.violinplot(x="cell_types", y="scale_eMB", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
               hue = "disease", hue_order=['NTT-N','EP-PE', ], palette = colors,
              inner = None)
sns.stripplot(x="cell_types", y="scale_eMB", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
              hue = "disease", hue_order=['NTT-N', 'EP-PE'],
              size= 0.5, color= "black", dodge = True,  alpha = 0.25)

ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.xlabel('')
plt.ylabel('Expressed rare variant burden (log1p)')
plt.ylim(-1.5, 10.1)
plt.legend().remove()

plt.savefig("output/figure4/Troph_eMB.pdf", format="pdf", bbox_inches="tight")
plt.show()


## Figure 4C

In [ ]:
df = pd.read_csv("input/single_cell/troph_cnv_20260920.csv")
df['cell_types'] = np.where(df['cell_type'] == "CTB", 'Cytotrophoblast', 'Extravillous trophoblast')

In [ ]:
df['cell_types'].value_counts()

In [ ]:
df['proportion_scaled_dupli_all'].describe()

In [ ]:
colors = ['#B1CCC2', '#6666A3']
ax = sns.violinplot(x="cell_types", y="proportion_scaled_dupli_all", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
               hue = "disease", hue_order=['NTT-N', 'EP-PE'], palette = colors,
              inner = None)
sns.stripplot(x="cell_types", y="proportion_scaled_dupli_all", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
              hue = "disease", hue_order=['NTT-N', 'EP-PE'],
              size= 0.5, color= "black", dodge = True,  alpha = 0.25)


ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.xlabel('')
plt.ylabel('Proportion of copy number variants')
plt.ylim(-0.03, 0.23)
plt.legend().remove()

plt.savefig("output/figure4/Troph_CNV.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Figure 4D

In [ ]:
df = pd.read_csv("input/single_cell/troph_UCell_20260920.csv")
df['cell_types'] = np.where(df['cell_type'] == "CTB", 'Cytotrophoblast', 'Extravillous trophoblast')
df['cell_types'].value_counts()

In [ ]:
df['signature_1'].describe()

In [ ]:
colors = ['#B1CCC2', '#6666A3']
ax = sns.violinplot(x="cell_types", y="signature_1", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
               hue = "disease", hue_order=['NTT-N', 'EP-PE'], palette = colors,
              inner = None)
sns.stripplot(x="cell_types", y="signature_1", data=df, order = ['Cytotrophoblast', 'Extravillous trophoblast'],
              hue = "disease", hue_order=['NTT-N', 'EP-PE'],
              size= 0.5, color= "black", dodge = True,  alpha = 0.25)


ax.tick_params(direction='out', length=2, width=1, colors='black', bottom=True, left=True)
ax.xaxis.label.set_color('black')
ax.yaxis.label.set_color('black')

plt.xlabel('')
plt.ylabel('Hypoxia response score')
plt.ylim(-0.04, 0.44)
plt.legend().remove()

plt.savefig("output/figure4/Troph_hypoxia.pdf", format="pdf", bbox_inches="tight")
plt.show()
